# ARC AGI 3 Colab Training

This notebook does four things:

1. Copies the project from Google Drive to the Colab local disk
2. Installs the required dependencies
3. Runs public environment collection and model training
4. Writes logs, validation outputs, and checkpoints back to `Training_Output/<timestamp>/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ARC Prize 2026 - ARC-AGI-3')
LOCAL_WORKDIR = Path('/content/ARC Prize 2026 - ARC-AGI-3')
RUN_TS = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'Training_Output' / RUN_TS
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Drive project root:', DRIVE_PROJECT_ROOT)
print('Local workdir:', LOCAL_WORKDIR)
print('Output root:', OUTPUT_ROOT)


In [ ]:
!rm -rf "$LOCAL_WORKDIR"
!mkdir -p "$LOCAL_WORKDIR"
!rsync -a --delete --exclude '.git' "$DRIVE_PROJECT_ROOT/" "$LOCAL_WORKDIR/"
%cd "$LOCAL_WORKDIR"


In [ ]:
import os, sys, subprocess

def run(cmd):
    print('>>>', cmd)
    subprocess.check_call(cmd, shell=True)

run('python -m pip install -U pip wheel setuptools')
run('python -m pip install -U torch torchvision torchaudio')

try:
    run('python -m pip install -U arc-agi==0.9.8 arcengine==0.9.3')
except Exception:
    print('PyPI install failed, trying local wheels...')
    run('python -m pip install arc_agi_3_wheels/*.whl')

run('python - <<\'PY\'\nimport torch\nprint("torch", torch.__version__)\nprint("cuda", torch.cuda.is_available())\nif torch.cuda.is_available():\n    print("device", torch.cuda.get_device_name(0))\nPY')


In [ ]:
HARDWARE_PROFILE = 'a100'  # Default starting profile. Switch to 'h100' only if needed.
COLLECT_STEPS = 96
CHECKPOINT_EVERY_STEPS = 100
RESUME_CHECKPOINT = None  # Example: OUTPUT_ROOT / 'checkpoints' / 'last.pth'

collect_cmd = f'''python -m src.collect \
  --project-root "{LOCAL_WORKDIR}" \
  --output-root "{OUTPUT_ROOT}" \
  --hardware-profile {HARDWARE_PROFILE} \
  --seeds 0,1,2,3 \
  --max-steps {COLLECT_STEPS}'''

run(collect_cmd)


In [ ]:
resume_arg = '' if RESUME_CHECKPOINT is None else f' \\\n  --resume "{RESUME_CHECKPOINT}"'

train_cmd = f'''python -m src.train \
  --project-root "{LOCAL_WORKDIR}" \
  --data "{OUTPUT_ROOT / 'collected' / 'episodes.jsonl.gz'}" \
  --output-dir "{OUTPUT_ROOT}" \
  --hardware-profile {HARDWARE_PROFILE} \
  --max-steps 192 \
  --online-val-games 5 \
  --checkpoint-every-steps {CHECKPOINT_EVERY_STEPS}{resume_arg}'''

run(train_cmd)


In [ ]:
eval_cmd = f'''python -m src.evaluate \
  --project-root "{LOCAL_WORKDIR}" \
  --checkpoint "{OUTPUT_ROOT / 'checkpoints' / 'best.pth'}" \
  --output "{OUTPUT_ROOT / 'public_eval.json'}" \
  --split val'''

run(eval_cmd)


In [ ]:
import pandas as pd
from pathlib import Path

metrics_path = OUTPUT_ROOT / 'metrics.csv'
display(pd.read_csv(metrics_path).tail())
print('Best checkpoint:', OUTPUT_ROOT / 'checkpoints' / 'best.pth')
print('Public eval:', OUTPUT_ROOT / 'public_eval.json')
